In [0]:
%run ./_utils

In [ ]:
import json
from datetime import datetime
from pyspark.sql import functions as F

#### Create `openalex.institutions.openalex_institutions_snapshot` in same format as API

In [ ]:
df_transformed = (
    spark.read.table("openalex.institutions.institutions_api")
    # Prefix numeric ID with full URL
    .withColumn("id", F.concat(F.lit("https://openalex.org/I"), F.col("id")))
    # Coalesce null arrays to empty arrays
    .withColumn("lineage", F.coalesce(F.col("lineage"), F.array()))
    .withColumn("display_name_acronyms", F.coalesce(F.col("display_name_acronyms"), F.array()))
    .withColumn("display_name_alternatives", F.coalesce(F.col("display_name_alternatives"), F.array()))
    .withColumn("roles", F.coalesce(F.col("roles"), F.array()))
    .withColumn("repositories", F.coalesce(F.col("repositories"), F.array()))
    .withColumn("topics", F.coalesce(F.col("topics"), F.array()))
    .withColumn("topic_share", F.coalesce(F.col("topic_share"), F.array()))
    .withColumn("associated_institutions", F.coalesce(F.col("associated_institutions"), F.array()))
    .withColumn("counts_by_year", F.coalesce(F.col("counts_by_year"), F.array()))
)

df_transformed.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("openalex.institutions.openalex_institutions_snapshot")

#### Export to JSONL + Parquet on S3
Writes both formats under `s3://openalex-snapshots/full/{date}/{format}/institutions/`.

In [0]:
date_str = get_snapshot_date()
df = spark.read.table("openalex.institutions.openalex_institutions_snapshot")
export_partitioned_all_formats(spark, dbutils, df, date_str, "institutions", salt=False)